### Welcome to Week 6 Day 3!

Let's experiment with a bunch more MCP Servers

In [1]:
from dotenv import load_dotenv
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, trace, set_default_openai_client, set_default_openai_api
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)
set_default_openai_client(AsyncAzureOpenAI(), use_for_tracing=False)
set_default_openai_api("chat_completions")

### The first type of MCP Server: runs locally, everything local

Here's a really interesting one: a knowledge-graph based memory.

It's a persistent memory store of entities, observations about them, and relationships between them.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory


In [2]:
params = {"command": "npx","args": ["-y", "mcp-memory-libsql"],"env": {"LIBSQL_URL": "file:./memory/ed.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', description='Create new entities with observations and optional embeddings', inputSchema={'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'entityType': {'type': 'string'}, 'observations': {'type': 'array', 'items': {'type': 'string'}}, 'embedding': {'type': 'array', 'items': {'type': 'number'}, 'description': 'Optional vector embedding for similarity search'}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, annotations=None),
 Tool(name='search_nodes', description='Search for entities and their relations using text or vector similarity', inputSchema={'type': 'object', 'properties': {'query': {'oneOf': [{'type': 'string', 'description': 'Text search query'}, {'type': 'array', 'items': {'type': 'number'}, 'description': 'Vector for similarity search'}]}}, 'required': ['query']}, annotations=None),
 Tool(name='read_graph', description='Get 

In [3]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-5-mini-2025-08-07"

In [4]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Nice to meet you, Ed — thanks for the context. I saved your name and that you’re an LLM engineer teaching a course on AI agents that covers the MCP protocol.

How can I help right now? Here are things I can prepare quickly (pick any or tell me what you prefer):
- Course/lecture outline and pacing for one class or a full course
- Slide decks or speaker notes for a lecture on MCP and agent-tool integration
- Hands-on labs: code examples, templates, and step-by-step MCP integrations
- Homework, quiz/exam questions, and project prompts (with evaluation rubrics)
- Demo scripts and datasets for in-class demos or notebooks
- Safety & evaluation checklist for agent/tool interactions

Quick clarifying Qs to tailor materials:
- Audience level (undergrad, grad, practitioners)?
- Course length or lecture duration?
- Preferred tech stack/languages (Python, JS, LangChain, etc.)?
- Do you want ready-to-run code (notebooks, Docker, templates)?

Tell me which item to start and any constraints, and I’ll produce it.

In [5]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))

I have a short memory entry for you. Current recorded facts:
- Name: Ed
- Profession: LLM engineer
- Teaching: a course about AI Agents, including the MCP protocol (for connecting agents with tools, resources and prompt templates)

Would you like to correct or add anything (e.g., preferred name, email, bio, interests, projects)? I can update, add more details, or delete this memory if you want.

### Check the trace:

https://platform.openai.com/traces

### The 2nd type of MCP server - runs locally, calls a web service

### Still use serper, as brave API requires credit card

Set up your account, and put your key in the .env under `SERPER_API_KEY`

In [6]:
env = {"SERPER_API_KEY": os.getenv("SERPER_API_KEY")}
params = {"command": "npx", "args": ["-y", "serper-search-scrape-mcp-server"], "env": env}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='google_search', description='Tool to perform web searches via Serper API and retrieve rich results. It is able to retrieve organic search results, people also ask, related searches, and knowledge graph.', inputSchema={'type': 'object', 'properties': {'q': {'type': 'string', 'description': 'Search query string'}, 'gl': {'type': 'string', 'description': "Optional region code for search results in ISO 3166-1 alpha-2 format (e.g., 'us')"}, 'hl': {'type': 'string', 'description': "Optional language code for search results in ISO 639-1 format (e.g., 'en')"}, 'location': {'type': 'string', 'description': "Optional location for search results (e.g., 'SoHo, New York, United States', 'California, United States')"}, 'num': {'type': 'number', 'description': 'Number of results to return (default: 10)'}, 'tbs': {'type': 'string', 'description': "Time-based search filter ('qdr:h' for past hour, 'qdr:d' for past day, 'qdr:w' for past week, 'qdr:m' for past month, 'qdr:y' for past year)"}, 

In [7]:
instructions = "You are able to search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. \
For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-5-mini-2025-08-07"

In [8]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Quick summary (as of 2025-09-11)

What’s happening now
- Price/market: AMZN is trading roughly in the low-to-mid $230s (recent closes around $230–238). Market cap ≈ $2.45–2.5T; 52‑week range ~ $161–$242.5. (Yahoo Finance)
- Recent moves/catalysts: shares fell after Amazon’s Q2 report (July 31) despite beats on EPS and revenue because AWS growth and conservative operating‑income guidance disappointed investors. More recently the stock got a lift from a JetBlue deal for Project Kuiper (satellite internet) and continued AI/robotics initiatives. (Investor’s Business Daily, MarketBeat)
- Analyst sentiment: strong buy skew — most sell‑side analysts remain bullish. Median 12‑month target area ≈ $260–$265, with a range of roughly $230 (low) to ~$300 (high). Several firms have raised targets in recent weeks. (FactSet/247WallSt/IBD)

Key positives
- Long-term growth drivers remain intact: AWS (cloud + enterprise AI), advertising, Prime/fulfillment scale, and new businesses (Kuiper, robotics, expanded grocery/same‑day). Analysts highlight AI investments and ad growth as upside catalysts.
- Institutional ownership remains high and many analysts have upgraded targets, supporting further upside if fundamentals/AI wins continue.

Key risks / headwinds
- AWS acceleration has lagged some competitors recently, which is a major short‑to‑medium term concern for investors.
- High ongoing capex for satellites/AI/robotics is pressuring free cash flow vs. historical levels.
- Macro/policy risks (tariff uncertainty, consumer spending) and occasional guidance conservatism can drive sharp short‑term volatility.
- Technicals: some technicians point to resistance around ~$240 (a possible “triple top”) — either a breakout above that level or a pullback is possible. (MarketBeat)

Valuation snapshot
- Trailing P/E ~35; forward P/E ~30 (Yahoo Finance). Analysts’ median target implies ~10–15% upside from current levels, but targets vary significantly.

Bottom line (brief outlook)
- Consensus short‑to‑medium‑term view is cautiously bullish: the company’s AI/cloud/ads/grocery initiatives provide credible upside, so many analysts rate AMZN a buy, but the stock remains vulnerable to volatility if AWS growth or guidance disappoints. If you’re trading, watch the ~$240 technical level for confirmation of a sustained breakout; if investing longer term, the fundamental case hinges on AWS/AI execution and capex payoff.

### As usual, check out the trace:

https://platform.openai.com/traces

## And now the third type: running remotely

It's actually really hard to find a "remote MCP server" aka "hosted MCP server" aka "managed MCP server".

It's not a common model for using or sharing MCP servers, and there isn't a standard way to discover remote MCP servers.

Anthropic lists some remote MCP servers, but these are for paid applications with business users:

https://docs.anthropic.com/en/docs/agents-and-tools/remote-mcp-servers

CloudFlare has tooling for you to create and deploy your own remote MCP servers, but this does not seem to be a common practice:

https://developers.cloudflare.com/agents/guides/remote-mcp-server/


# And back to the 2nd type: the Polygon.io MCP Server

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">PLEASE READ!!-</h2>
            <span style="color:#ff7800;">This service for financial market data has both a FREE plan and a PAID plan, and we can use either depending on your appetite.
            </span>
        </td>
    </tr>
</table>

## NEW SECTION: Introducing polygon.io

Polygon.io is a hugely popular financial data provider. It has a free plan and a paid plan. And it also has an MCP Server!

First, read up on polygon.io on their excellent website, including looking at their pricing:

https://polygon.io

### Polygon.io Part 1: Polygon.io free service (the paid will be totally optional, of course!)

1. Please sign up for polygon.io (top right)  
2. Once signed in, please select "Keys" in the left hand navigation
3. Press the blue "New Key" button
4. Copy the key name
5. Edit your .env file and add the row:

`POLYGON_API_KEY=xxxx`

In [9]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY is not set")

In [10]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

PreviousCloseAgg(ticker='AAPL', close=226.79, high=232.42, low=225.95, open=232.185, timestamp=1757534400000, volume=83440810.0, vwap=227.7892)

### Wrapped into a python module that caches end of day prices

I've made a python module `market.py` that uses this API to look up share prices.

But the free API is quite heavily rate limited - so I've been a bit sneaky; when you ask for a share price, this function retrieves the entire end-of-day equity market, and caches it in our database.


In [11]:
from market import get_share_price
get_share_price("AAPL")

226.79

In [12]:
# no rate limiting concerns!

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

226.79

### And I've made this into an MCP Server

Just as we did with accounts.py; see `market_server.py`

In [13]:
params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

[Tool(name='lookup_share_price', description='This tool provides the current price of the given stock symbol.\n\n    Args:\n        symbol: the symbol of the stock\n    ', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'}, annotations=None)]

### Let's try it out!

Hopefully gpt-4o-mini is smart enough to know that the symbol for Apple is AAPL

In [14]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple?"
model = "gpt-5-mini-2025-08-07"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Apple (AAPL): $226.79 per share (USD). This is a snapshot — prices change frequently. Would you like a live update, intraday chart, or recent news?

## Polygon.io Part 2: Paid Plan - Totally Optional!

If you are interested, you can subscribe to the monthly plan to get more up to date market data, and unlimited API calls.

If you do wish to do this, then it also makes sense to use the full MCP server that Polygon.io has released, to take advantage of all their functionality.



In [ ]:

params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@v0.1.0", "mcp_polygon"],
          "env": {"POLYGON_API_KEY": polygon_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools


### Wow that's a lot of tools!

Let's try them out - hopefully the sheer number of tools doesn't overwhelm gpt-4o-mini!

With the $29 monthly plan, we don't have access to some of the APIs, so I've needed to specify which APIs can be called.

If you've splashed out on a bigger plan, feel free to remove my extra constraint..

In [ ]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple? Use your get_snapshot_ticker tool to get the latest price."
model = "gpt-5-mini-2025-08-07"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Setting up your .env file

If you do decide to have a paid plan, please add this to your .env file to indicate:

`POLYGON_PLAN=paid`

And if you decide to go all the way for the realtime API, then please do:

`POLYGON_PLAN=realtime`

In [ ]:
load_dotenv(override=True)

polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

if is_paid_polygon:
    print("You've chosen to subscribe to the paid Polygon plan, so the code will look at prices on a 15 min delay")
elif is_realtime_polygon:
    print("Wowzer - you've chosen to subscribe to the realtime Polygon plan, so the code will look at realtime prices")
else:
    print("According to your .env file, you've chosen to subscribe to the free Polygon plan, so the code will look at EOD prices")

## And that's it for today!

I've removed the part of this lab that uses the "Financial Datasets" mcp server, because it's inferior - more expensive with fewer APIs.

And this way we get to use the same provider for Free and Paid APIs.

But if you want to see the code, just look in the git history for a prior version.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Explore MCP server marketplaces and integrate your own, using all 3 approaches.
            </span>
        </td>
    </tr>
</table>